# pipe_catedra/05 — Pipe único: de los crudos al submit, en un solo notebook resumible

Inspirado en `pipe_unico` de un companero de cursada (mismo estilo: una
sola celda de "palancas" controla todo, cada etapa escribe su checkpoint
y se saltea si ya existe, asi que si la VM muere a mitad de camino
-- spot VM, GCS-FUSE que se cuelga, lo que sea -- correr el notebook de
nuevo retoma donde quedo en vez de arrancar de cero).

No reemplaza a `01_`/`02_`/`02_FE_bis`/`03_`/`04_` -- **reusa exactamente
los mismos archivos de cache** (mismos nombres, misma logica portada de
esos cuatro notebooks) asi que si ya corriste alguno de ellos por
separado, este notebook encuentra esas cachés y las saltea. Lo nuevo es
que las 5 etapas (preprocesamiento -> FE base -> FE avanzado opcional ->
Optuna -> entrenamiento final + submit) corren en cadena, desde una sola
celda de configuracion, sin tener que abrir 4 notebooks distintos.

**La palanca `usar_fe_avanzado`** decide si se suma tu feature engineering
extra (`02_FE_bis`: vecinos sustitutos/complementarios, hojas de Random
Forest, poda por shadow features) encima del FE base de la catedra, o si
se usa solo el FE base. Por default esta en `True`.

**El nombre del experimento se arma solo** a partir de las palancas que
importan para Optuna/entrenamiento final (`tipo_target`, `objective_lgbm`,
`regularizacion`, `esquema_val`, `usar_fe_avanzado`, cantidad de semillas
del ensemble) -- a proposito, para evitar el bug que documento el
companero en su bitacora: si el nombre de experimento no cambia solo al
cambiar una palanca relevante, una corrida nueva puede encontrar el
checkpoint viejo y devolver silenciosamente el resultado de la corrida
anterior sin avisar.


## 0) Setup


In [ ]:
import os, shutil, subprocess, time, json
from pathlib import Path

import polars as pl
import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna
from sklearn.ensemble import RandomForestRegressor
from tqdm.auto import tqdm

optuna.logging.set_verbosity(optuna.logging.WARNING)


def resolver_bucket() -> Path:
    """VM de la catedra -> ~/buckets/b1 | Colab -> /content/buckets/b1 | local -> env."""
    env = os.environ.get("LABO3_BUCKET")
    if env and Path(env).expanduser().exists():
        return Path(env).expanduser().resolve()
    for cand in (Path.home() / "buckets" / "b1",
                 "/content/buckets/b1",
                 "/home/ds/buckets/b1"):
        if Path(cand).is_dir():
            return Path(cand)
    raise RuntimeError(
        "No encontre el bucket. Defini LABO3_BUCKET, ej: "
        "os.environ['LABO3_BUCKET'] = '/home/usuario/labo3-bucket'"
    )


BUCKET  = resolver_bucket()
DIR_RAW = BUCKET / "datasets"
DIR_OUT = BUCKET / "exp_pipe_catedra"
DIR_RAW.mkdir(parents=True, exist_ok=True)
DIR_OUT.mkdir(parents=True, exist_ok=True)
print(f"BUCKET: {BUCKET}")
print(f"crudos: {DIR_RAW}")
print(f"salida: {DIR_OUT}")

# Kaggle auth
kaggle_dst = Path.home() / ".kaggle" / "kaggle.json"
kaggle_dst.parent.mkdir(parents=True, exist_ok=True)
if kaggle_dst.exists():
    kaggle_dst.chmod(0o600)
    print("Kaggle auth OK (ya estaba en ~/.kaggle)")
else:
    _encontrado = False
    for cand in (BUCKET / "kaggle.json", BUCKET / "kaggle" / "kaggle.json"):
        if cand.exists():
            shutil.copy(cand, kaggle_dst)
            kaggle_dst.chmod(0o600)
            print(f"Kaggle auth OK (copiado de {cand})")
            _encontrado = True
            break
    if not _encontrado:
        print("aviso: kaggle.json no encontrado. Subilo a ~/.kaggle/kaggle.json o al bucket.")


def descargar(archivo):
    url = f"https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/{archivo}"
    dst = DIR_RAW / archivo
    if not dst.exists():
        subprocess.run(["wget", url, "-O", str(dst)], check=True)
    print(f"ok: {archivo}")


for _a in ("sell-in.txt.gz", "tb_productos.txt", "tb_stocks.txt", "product_id_apredecir201912.txt"):
    if (DIR_RAW / _a).exists():
        print(f"ya existe: {_a}")
    else:
        descargar(_a)


def guardar_parquet_atomico(df, path):
    """Escribe a .tmp y renombra -- si la VM muere a mitad de la escritura,
    el archivo final nunca queda a medio escribir (uno de los dolores de
    cabeza mas comunes con spot VMs, documentado por el companero)."""
    path = Path(path)
    tmp = path.with_suffix(path.suffix + '.tmp')
    df.write_parquet(tmp)
    tmp.replace(path)


def a_indice_mes(p: int) -> int:
    return (p // 100) * 12 + (p % 100) - 1


def desplazar_meses(p: int, k: int) -> int:
    m = a_indice_mes(p) + k
    return (m // 12) * 100 + (m % 12) + 1


## 1) Palancas — toda la configuracion en un solo lugar


In [ ]:
PARAM = {
    # ── Etapa 1: preprocesamiento ──────────────────────────────────────
    'modo_agrupacion': 'producto',
    'solo_predecir': True,

    # ── Etapa 2: FE base (catedra) ───────────────────────────────────────
    'lags': list(range(0, 12)),
    'ventana_rolling': None,
    'incluir_market_share': True,
    'horizonte': 2,

    # ── Etapa 2bis: FE avanzado -- TU feature engineering extra ──────────
    # (vecinos sustitutos/complementarios, hojas de Random Forest, poda
    # por shadow features -- ver 02_FE_bis.ipynb)
    'usar_fe_avanzado': True,
    'n_vecinos': 3,
    'mes_corte_avanzado': None,
    'meses_atras_default': 24,
    'n_arboles_rf': 50,
    'profundidad_rf': 6,
    'min_hoja_rf': 50,
    'n_shadow': 10,
    'n_repeticiones_shadow': 5,
    'umbral_mayoria_shadow': 0.5,

    # ── Etapa 3: Optuna ──────────────────────────────────────────────────
    'n_trials': 50,
    'esquema_val': 'febreros',
    'walk_forward_k': 3,
    'mes_objetivo': 2,
    'metrica': 'wape',
    'sampling_frac': None,
    'objective_lgbm': 'regression',
    'tweedie_optimizar': True,
    'regularizacion': 'normal',
    'decay_recencia': None,
    'tipo_target': 'nivel',
    'features_excluir': [],

    # ── Etapa 4: entrenamiento final + submit ────────────────────────────
    'desescalar': False,
    'clip_min': 0.0,
    'semillas_ensemble': [102191],
    'submit': False,
    'kaggle_competition': 'labo-iii-2026-rosario',

    'semilla': 102191,
    'cols_categoricas': ['cat1', 'cat2', 'cat3', 'brand'],

    # ── Sufijo libre para distinguir corridas identicas (ej. repetir con
    # otra semilla) sin tocar las palancas de arriba ──────────────────────
    'sufijo_experimento': '',

    # ── Forzar recomputo de una etapa aunque ya tenga checkpoint ─────────
    # valores posibles: 'preprocesamiento', 'fe', 'fe_avanzado', 'optuna', 'final'
    'forzar': set(),
}

MODO = PARAM['modo_agrupacion']

# El nombre de experimento se arma SOLO a partir de las palancas que le
# importan a Optuna/entrenamiento final -- evita el bug de "cambie una
# palanca pero el checkpoint viejo se reusa sin avisar" (documentado por
# el companero en su bitacora).
_tag_fe = 'feAvanzado' if PARAM['usar_fe_avanzado'] else 'feBase'
_partes_experimento = [
    PARAM['tipo_target'], PARAM['objective_lgbm'], PARAM['regularizacion'],
    PARAM['esquema_val'], _tag_fe, f"ens{len(PARAM['semillas_ensemble'])}",
]
if PARAM['sufijo_experimento']:
    _partes_experimento.append(PARAM['sufijo_experimento'])
PARAM['experimento'] = '__'.join(str(p) for p in _partes_experimento)

_tag_solo = 'solo780' if PARAM['solo_predecir'] else 'todosProductos'
PARAM['path_pre']          = str(DIR_OUT / f"z301_preprocessed_{MODO}_{_tag_solo}.parquet")
PARAM['path_fe']           = str(DIR_OUT / f"z302_features_{MODO}_{_tag_solo}.parquet")
PARAM['path_fe_infer']     = str(DIR_OUT / f"z302_inferencia_{MODO}_{_tag_solo}.parquet")
PARAM['path_fe_bis']       = str(DIR_OUT / f"z302bis_features_{MODO}_{_tag_solo}.parquet")
PARAM['path_fe_bis_infer'] = str(DIR_OUT / f"z302bis_inferencia_{MODO}_{_tag_solo}.parquet")
PARAM['path_hiper']        = str(DIR_OUT / f"z303_hiper_{MODO}_{PARAM['experimento']}.json")
_db_name = f"z303_optuna_{MODO}_{PARAM['experimento']}.db"
PARAM['path_storage']      = f"sqlite:///{Path.home() / _db_name}"
PARAM['study_name']        = f"{PARAM['experimento']}_{MODO}"
PARAM['path_submit']       = str(DIR_OUT / f"z304_predicciones_{MODO}_{PARAM['experimento']}.csv")

print(f"experimento: {PARAM['experimento']}")
print('Parametros:', PARAM)


## 2) Etapa 1 — Preprocesamiento (identico a `01_Preprocesamiento.ipynb`)


In [ ]:
t0 = time.time()
if Path(PARAM['path_pre']).exists() and 'preprocesamiento' not in PARAM['forzar']:
    print(f"[preprocesamiento] cache encontrada -> se saltea: {PARAM['path_pre']}")
    df_full = pl.read_parquet(PARAM['path_pre'])
else:
    print('[preprocesamiento] calculando...')
    import duckdb

    df_raw = pl.read_csv(
        DIR_RAW / 'sell-in.txt.gz', separator='\t',
        schema_overrides={
            'periodo': pl.Int32, 'customer_id': pl.Int32, 'product_id': pl.Int32,
            'cust_request_qty': pl.Int32, 'cust_request_tn': pl.Float32, 'tn': pl.Float32,
        }
    )
    tb_apredecir = pl.read_csv(DIR_RAW / 'product_id_apredecir201912.txt', separator='\t',
                               schema_overrides={'product_id': pl.Int32})
    tb_productos = pl.read_csv(DIR_RAW / 'tb_productos.txt', separator='\t')

    prod_cat = tb_productos.select(['product_id', 'cat1', 'cat2', 'cat3', 'brand']).with_columns(
        pl.col('product_id').cast(pl.Int32))
    df_univ = (df_raw.group_by(['product_id', 'periodo']).agg(pl.col('tn').sum().alias('tn'))
                     .join(prod_cat, on='product_id', how='left'))

    periodos_u = sorted(df_univ['periodo'].unique().to_list())
    p2i = {p: i for i, p in enumerate(periodos_u)}
    periodos_idx = pl.DataFrame({'periodo': periodos_u, 'idx': list(range(len(periodos_u)))}) \
                     .with_columns(pl.col('periodo').cast(pl.Int32), pl.col('idx').cast(pl.Int32))

    con = duckdb.connect()
    con.register('df_univ', df_univ)
    con.register('periodos_idx', periodos_idx)

    totales = {}
    for nivel in ['cat1', 'cat2', 'cat3', 'brand']:
        totales[nivel] = (df_univ.group_by([nivel, 'periodo'])
                                 .agg(pl.col('tn').sum().alias(f'tn_total_{nivel}')))
    activos = {}
    for nivel in ['cat2', 'cat3']:
        activos[nivel] = (df_univ.filter(pl.col('tn') > 0).group_by([nivel, 'periodo'])
                                 .agg(pl.col('product_id').n_unique().alias(f'productos_activos_{nivel}')))
    nac = (df_univ.filter(pl.col('tn') > 0).group_by('product_id')
                 .agg(pl.col('periodo').min().alias('nacimiento'))
                 .join(prod_cat, on='product_id', how='left'))
    con.register('nac', nac)

    def nacimientos_por_nivel(nivel):
        q = f"""
            SELECT v."{nivel}" AS "{nivel}", pi.periodo,
                   COUNT(n.product_id) AS "productos_nuevos_{nivel}_3m"
            FROM (SELECT DISTINCT "{nivel}" FROM df_univ) v
            CROSS JOIN periodos_idx pi
            LEFT JOIN (
                SELECT nac.product_id, nac."{nivel}" AS "{nivel}", pin.idx AS nac_idx
                FROM nac JOIN periodos_idx pin ON pin.periodo = nac.nacimiento
            ) n
              ON n."{nivel}" IS NOT DISTINCT FROM v."{nivel}"
             AND n.nac_idx BETWEEN pi.idx - 2 AND pi.idx
            GROUP BY v."{nivel}", pi.periodo
        """
        return con.sql(q).pl().with_columns(pl.col('periodo').cast(pl.Int32))

    nuevos_cat2 = nacimientos_por_nivel('cat2')
    nuevos_cat3 = nacimientos_por_nivel('cat3')

    if PARAM['solo_predecir']:
        df_raw = df_raw.join(tb_apredecir, on='product_id', how='inner')

    if PARAM['modo_agrupacion'] == 'producto':
        df_agrupado = (df_raw.group_by(['product_id', 'periodo']).agg(pl.col('tn').sum().alias('tn'))
                             .with_columns(pl.col('product_id').cast(pl.Int64).alias('agrupa_id'),
                                          pl.lit(None).cast(pl.Int32).alias('customer_id')))
    elif PARAM['modo_agrupacion'] == 'cliente_producto':
        MULTIPLICADOR_AGRUPA = 100_000
        df_agrupado = (df_raw.group_by(['customer_id', 'product_id', 'periodo']).agg(pl.col('tn').sum().alias('tn'))
                             .with_columns((pl.col('customer_id').cast(pl.Int64) * MULTIPLICADOR_AGRUPA
                                          + pl.col('product_id').cast(pl.Int64)).alias('agrupa_id')))
    else:
        raise ValueError(f"modo_agrupacion invalido: {PARAM['modo_agrupacion']}")
    df_agrupado = df_agrupado.sort(['agrupa_id', 'periodo'])

    periodos_all = sorted(df_agrupado['periodo'].unique().to_list())
    periodo_a_idx = {p: i for i, p in enumerate(periodos_all)}
    idx_a_periodo = {i: p for p, i in periodo_a_idx.items()}
    periodos_all_idx = pl.DataFrame({'periodo': periodos_all, 'idx': list(range(len(periodos_all)))}) \
                         .with_columns(pl.col('periodo').cast(pl.Int32), pl.col('idx').cast(pl.Int32))

    ventas_positivas = df_agrupado.filter(pl.col('tn') > 0)
    primeros = ventas_positivas.group_by('agrupa_id').agg(pl.col('periodo').min().alias('primer_periodo_activo'))
    ultimos = ventas_positivas.group_by('agrupa_id').agg(pl.col('periodo').max().alias('ultimo_periodo_activo'))
    primeros_ultimos = primeros.join(ultimos, on='agrupa_id', how='left')

    con.register('primeros_ultimos', primeros_ultimos)
    con.register('periodos_all_idx', periodos_all_idx)
    grid = con.sql("""
        SELECT pu.agrupa_id, pi.periodo
        FROM primeros_ultimos pu
        JOIN periodos_all_idx pi_ini ON pi_ini.periodo = pu.primer_periodo_activo
        JOIN periodos_all_idx pi_fin ON pi_fin.periodo = pu.ultimo_periodo_activo
        JOIN periodos_all_idx pi ON pi.idx BETWEEN pi_ini.idx AND pi_fin.idx
    """).pl().with_columns([pl.col('agrupa_id').cast(pl.Int64), pl.col('periodo').cast(pl.Int32)])

    df_full = (grid.join(df_agrupado.select(['agrupa_id', 'periodo', 'product_id', 'customer_id', 'tn']),
                        on=['agrupa_id', 'periodo'], how='left')
                   .with_columns(pl.col('tn').fill_null(0.0))
                   .sort(['agrupa_id', 'periodo']))

    if PARAM['modo_agrupacion'] == 'producto':
        df_full = df_full.with_columns(pl.col('agrupa_id').cast(pl.Int32).alias('product_id'))
    else:
        MULTIPLICADOR_AGRUPA = 100_000
        df_full = df_full.with_columns([
            (pl.col('agrupa_id') // MULTIPLICADOR_AGRUPA).cast(pl.Int32).alias('customer_id'),
            (pl.col('agrupa_id') % MULTIPLICADOR_AGRUPA).cast(pl.Int32).alias('product_id'),
        ])

    tb_productos = tb_productos.with_columns(pl.col('product_id').cast(pl.Int32))
    df_full = df_full.join(tb_productos, on='product_id', how='left')

    df_cust = (df_raw.group_by(['product_id', 'customer_id', 'periodo']).agg(pl.col('tn').sum().alias('tn_cust'))
                     .filter(pl.col('tn_cust') > 0))
    df_prod_tot = df_cust.group_by(['product_id', 'periodo']).agg(pl.col('tn_cust').sum().alias('tn_prod_tot'))
    df_n_cust = df_cust.group_by(['product_id', 'periodo']).agg(pl.col('customer_id').n_unique().alias('clientes_activos'))
    df_cust = df_cust.join(df_prod_tot, on=['product_id', 'periodo'], how='left').with_columns(
        (pl.col('tn_cust') / pl.col('tn_prod_tot')).alias('share'))
    df_conc = df_cust.group_by(['product_id', 'periodo']).agg([
        pl.col('share').max().alias('cliente_principal_share'),
        (pl.col('share') ** 2).sum().alias('hhi_clientes'),
    ])
    df_cust_feats = df_n_cust.join(df_conc, on=['product_id', 'periodo'], how='left').with_columns([
        pl.col('clientes_activos').cast(pl.Int32),
        pl.col('cliente_principal_share').cast(pl.Float32),
        pl.col('hhi_clientes').cast(pl.Float32),
    ])
    df_full = df_full.join(df_cust_feats, on=['product_id', 'periodo'], how='left').with_columns([
        pl.col('clientes_activos').fill_null(0),
        pl.col('cliente_principal_share').fill_null(0.0),
        pl.col('hhi_clientes').fill_null(0.0),
    ])

    for nivel in ['cat1', 'cat2', 'cat3', 'brand']:
        df_full = df_full.join(totales[nivel], on=[nivel, 'periodo'], how='left')
    for nivel in ['cat2', 'cat3']:
        df_full = df_full.join(activos[nivel], on=[nivel, 'periodo'], how='left')
    df_full = df_full.join(nuevos_cat2, on=['cat2', 'periodo'], how='left')
    df_full = df_full.join(nuevos_cat3, on=['cat3', 'periodo'], how='left')
    df_full = df_full.with_columns([
        pl.col('tn_total_cat1').fill_null(0.0), pl.col('tn_total_cat2').fill_null(0.0),
        pl.col('tn_total_cat3').fill_null(0.0), pl.col('tn_total_brand').fill_null(0.0),
        pl.col('productos_activos_cat2').fill_null(0), pl.col('productos_activos_cat3').fill_null(0),
        pl.col('productos_nuevos_cat2_3m').fill_null(0), pl.col('productos_nuevos_cat3_3m').fill_null(0),
    ])
    df_full = df_full.with_columns([
        pl.when(pl.col('tn_total_cat2') > 0).then(pl.col('tn') / pl.col('tn_total_cat2')).otherwise(pl.lit(0.0)).alias('market_share_cat2'),
        pl.when(pl.col('tn_total_cat3') > 0).then(pl.col('tn') / pl.col('tn_total_cat3')).otherwise(pl.lit(0.0)).alias('market_share_cat3'),
        pl.when(pl.col('tn_total_brand') > 0).then(pl.col('tn') / pl.col('tn_total_brand')).otherwise(pl.lit(0.0)).alias('market_share_brand'),
    ])
    df_full = df_full.with_columns(pl.col('market_share_cat2').alias('market_share'))
    df_full = df_full.with_columns([
        pl.lit(PARAM['modo_agrupacion']).alias('modo_agrupacion'),
        pl.lit(PARAM['solo_predecir']).alias('solo_predecir'),
    ])

    guardar_parquet_atomico(df_full, PARAM['path_pre'])
    con.close()

print(f'[preprocesamiento] {df_full.height:,} filas x {len(df_full.columns)} columnas.   [{time.time()-t0:.0f}s]')


## 3) Etapa 2 — FE base (identico a `02_FE.ipynb`)


In [ ]:
t0 = time.time()
if Path(PARAM['path_fe']).exists() and Path(PARAM['path_fe_infer']).exists() and 'fe' not in PARAM['forzar']:
    print(f"[FE base] cache encontrada -> se saltea: {PARAM['path_fe']}")
    df_train_2 = pl.read_parquet(PARAM['path_fe'])
    df_infer_2 = pl.read_parquet(PARAM['path_fe_infer'])
else:
    print('[FE base] calculando...')
    dff = df_full

    for col in PARAM['cols_categoricas']:
        if col in dff.columns:
            dff = dff.with_columns(pl.col(col).cast(pl.Categorical).to_physical().cast(pl.Int32).alias(col))
    if 'descripcion' in dff.columns:
        dff = dff.drop('descripcion')

    dff = dff.sort(['agrupa_id', 'periodo'])
    dff = dff.with_columns([pl.col('tn').shift(k).over('agrupa_id').alias(f'lag_{k}') for k in PARAM['lags']])

    dff = dff.with_columns([
        (pl.col('lag_0') - pl.col('lag_1')).alias('delta_1'),
        (pl.col('lag_0') - pl.col('lag_2')).alias('delta_2'),
        (pl.col('lag_0') - pl.col('lag_3')).alias('delta_3'),
        (pl.col('lag_0') - pl.col('lag_11')).alias('delta_anual'),
    ])
    dff = dff.with_columns(((pl.col('lag_0') - pl.col('lag_2')) / 2).alias('pendiente_3m'))
    dff = dff.with_columns([
        pl.when(pl.col('lag_1') > 0).then(pl.col('lag_0') / pl.col('lag_1')).otherwise(pl.lit(0.0)).alias('ratio_0_1'),
        pl.when(pl.col('lag_3') > 0).then(pl.col('lag_0') / pl.col('lag_3')).otherwise(pl.lit(0.0)).alias('ratio_0_3'),
    ])
    dff = dff.with_columns([
        pl.col('tn').shift(1).rolling_mean(window_size=3, min_periods=1).over('agrupa_id').alias('media_movil_3'),
        pl.col('tn').shift(1).rolling_mean(window_size=6, min_periods=1).over('agrupa_id').alias('media_movil_6'),
    ])
    if 'market_share' in dff.columns:
        dff = dff.with_columns((pl.col('market_share') - pl.col('market_share').shift(3).over('agrupa_id')).alias('ms_delta_3'))

    dff = dff.with_columns(pl.int_range(1, pl.len() + 1).over('agrupa_id').cast(pl.Int32).alias('meses_activo'))
    dff = dff.with_columns((pl.col('tn') == pl.col('tn').shift(1).over('agrupa_id')).cast(pl.Int8).fill_null(0).alias('compro_igual_mes_ant'))
    es_cero = (pl.col('tn') == 0).cast(pl.Int32)
    dff = dff.with_columns(es_cero.alias('_es_cero'))
    dff = dff.with_columns((1 - pl.col('_es_cero')).cum_sum().over('agrupa_id').alias('_bloque'))
    dff = dff.with_columns(pl.col('_es_cero').cum_sum().over(['agrupa_id', '_bloque']).cast(pl.Int32).alias('racha_ceros'))
    sube = (pl.col('tn') > pl.col('tn').shift(1)).over('agrupa_id')
    baja = (pl.col('tn') < pl.col('tn').shift(1)).over('agrupa_id')
    dff = dff.with_columns([sube.cast(pl.Int8).fill_null(0).alias('_sube'), baja.cast(pl.Int8).fill_null(0).alias('_baja')])
    dff = dff.with_columns([(1 - pl.col('_sube')).cum_sum().over('agrupa_id').alias('_blq_sube'),
                            (1 - pl.col('_baja')).cum_sum().over('agrupa_id').alias('_blq_baja')])
    dff = dff.with_columns([
        pl.col('_sube').cum_sum().over(['agrupa_id', '_blq_sube']).cast(pl.Int32).alias('racha_crece'),
        pl.col('_baja').cum_sum().over(['agrupa_id', '_blq_baja']).cast(pl.Int32).alias('racha_cae'),
    ])
    dff = dff.with_columns((pl.col('_es_cero').cum_sum().over('agrupa_id') /
                            pl.int_range(1, pl.len() + 1).over('agrupa_id')).cast(pl.Float32).alias('pct_ceros_hist'))
    dff = dff.drop(['_es_cero', '_bloque', '_sube', '_baja', '_blq_sube', '_blq_baja'])

    ventana = PARAM['ventana_rolling']
    w = len(dff) if ventana is None else ventana
    dff = dff.with_columns([
        pl.col('tn').shift(1).rolling_mean(window_size=w, min_periods=1).over('agrupa_id').alias('media_rolling'),
        pl.col('tn').shift(1).rolling_std(window_size=w, min_periods=2).over('agrupa_id').alias('std_rolling'),
    ])
    dff = dff.with_columns(pl.when(pl.col('media_rolling') > 0).then(pl.col('tn') / pl.col('media_rolling'))
                           .otherwise(pl.lit(0.0)).alias('tn_scaled'))

    dff = dff.with_columns([(pl.col('periodo') % 100).cast(pl.Int8).alias('mes'),
                            (pl.col('periodo') // 100).cast(pl.Int16).alias('anio')])

    h = PARAM['horizonte']
    dff = dff.with_columns(pl.col('tn').shift(-h).over('agrupa_id').alias('tn_t2'))
    dff = dff.with_columns([pl.col('tn_t2').alias('target_nivel'), (pl.col('tn_t2') - pl.col('tn')).alias('target_delta')])

    df_train_2 = dff.filter(pl.col('tn_t2').is_not_null())
    df_infer_2 = dff.filter(pl.col('tn_t2').is_null())

    guardar_parquet_atomico(df_train_2, PARAM['path_fe'])
    guardar_parquet_atomico(df_infer_2, PARAM['path_fe_infer'])

print(f'[FE base] train {df_train_2.shape}   inferencia {df_infer_2.shape}   [{time.time()-t0:.0f}s]')


## 4) Etapa 2bis — FE avanzado (TU feature engineering, opcional segun `usar_fe_avanzado`)

Identico a `02_FE_bis.ipynb`: vecinos sustitutos/complementarios, hojas de
Random Forest, poda por shadow features en dos rondas. Si
`usar_fe_avanzado=False` se usa directamente el FE base de la etapa 2.


In [ ]:
t0 = time.time()
if not PARAM['usar_fe_avanzado']:
    print('[FE avanzado] usar_fe_avanzado=False -> se usa el FE base tal cual.')
    df_train_final = df_train_2
    df_infer_final = df_infer_2
elif Path(PARAM['path_fe_bis']).exists() and Path(PARAM['path_fe_bis_infer']).exists() and 'fe_avanzado' not in PARAM['forzar']:
    print(f"[FE avanzado] cache encontrada -> se saltea: {PARAM['path_fe_bis']}")
    df_train_final = pl.read_parquet(PARAM['path_fe_bis'])
    df_infer_final = pl.read_parquet(PARAM['path_fe_bis_infer'])
else:
    print('[FE avanzado] calculando...')
    df_c = pl.concat([
        df_train_2.with_columns(pl.lit(False).alias('_es_infer')),
        df_infer_2.with_columns(pl.lit(True).alias('_es_infer')),
    ], how='diagonal_relaxed')

    ULTIMO_PERIODO = int(df_c['periodo'].max())
    if PARAM['mes_corte_avanzado'] is None:
        _corte = desplazar_meses(ULTIMO_PERIODO, -PARAM['meses_atras_default'])
    else:
        _corte = PARAM['mes_corte_avanzado']
    print(f'  ultimo periodo: {ULTIMO_PERIODO}   corte vecinos/RF: periodo < {_corte}')

    # --- vecinos sustitutos/complementarios (a nivel producto) ---
    tot_prod = df_c.group_by(['product_id', 'periodo']).agg(pl.col('tn').sum().alias('tn_prod'))
    wide = (tot_prod.filter(pl.col('periodo') < _corte)
                    .pivot(on='product_id', index='periodo', values='tn_prod')
                    .sort('periodo').drop('periodo'))
    corr = wide.to_pandas().corr(method='spearman')
    N_VEC = PARAM['n_vecinos']
    vecinos_rows = []
    for p in corr.columns:
        s = corr[p].drop(labels=[p]).dropna()
        if s.empty:
            continue
        for vecino, r in s.sort_values().head(N_VEC).items():
            vecinos_rows.append({'product_id': p, 'tipo': 'sustituto', 'vecino_id': vecino, 'corr': float(r)})
        for vecino, r in s.sort_values(ascending=False).head(N_VEC).items():
            vecinos_rows.append({'product_id': p, 'tipo': 'complementario', 'vecino_id': vecino, 'corr': float(r)})
    vecinos = pl.from_pandas(pd.DataFrame(vecinos_rows)).with_columns(
        [pl.col('product_id').cast(pl.Int32), pl.col('vecino_id').cast(pl.Int32)])
    feat_vecinos = (vecinos.join(tot_prod.rename({'product_id': 'vecino_id', 'tn_prod': 'tn_vecino'}), on='vecino_id', how='left')
                          .group_by(['product_id', 'tipo', 'periodo']).agg(pl.col('tn_vecino').mean().alias('tn_vecino_prom')))
    feat_vecinos_piv = feat_vecinos.pivot(on='tipo', index=['product_id', 'periodo'], values='tn_vecino_prom')
    for col_falt in ('sustituto', 'complementario'):
        if col_falt not in feat_vecinos_piv.columns:
            feat_vecinos_piv = feat_vecinos_piv.with_columns(pl.lit(None, dtype=pl.Float64).alias(col_falt))
    feat_vecinos_piv = feat_vecinos_piv.rename({'sustituto': 'tn_sustitutos_prom', 'complementario': 'tn_complementarios_prom'})
    df_c = (df_c.join(feat_vecinos_piv, on=['product_id', 'periodo'], how='left')
                .with_columns([pl.col('tn_sustitutos_prom').fill_null(0.0), pl.col('tn_complementarios_prom').fill_null(0.0)]))
    print(f'  vecinos agregados.   [{time.time()-t0:.0f}s]')

    # --- hojas de Random Forest ---
    COLS_ID = ['agrupa_id', 'product_id', 'customer_id', 'periodo', '_es_infer']
    COLS_TARGET = ['tn', 'tn_t2', 'target_nivel', 'target_delta']
    COLS_META = ['modo_agrupacion', 'solo_predecir']
    PROHIBIDAS_BASE = set(COLS_ID) | set(COLS_TARGET) | set(COLS_META)
    FEATURES_BASE = [c for c in df_c.columns if c not in PROHIBIDAS_BASE]
    CAT_BASE = [c for c in PARAM['cols_categoricas'] if c in FEATURES_BASE]
    NUM_RF = [c for c in FEATURES_BASE if c not in CAT_BASE]

    PARAM_RF = dict(n_estimators=PARAM['n_arboles_rf'], max_depth=PARAM['profundidad_rf'],
                   min_samples_leaf=PARAM['min_hoja_rf'], n_jobs=-1, random_state=PARAM['semilla'])
    _tr_rf = df_c.filter((pl.col('periodo') < _corte) & pl.col('target_nivel').is_not_null())
    X_rf_train = _tr_rf.select(NUM_RF).fill_null(0.0).to_numpy()
    y_rf_train = _tr_rf['target_nivel'].fill_null(0.0).to_numpy()
    rf = RandomForestRegressor(**PARAM_RF)
    rf.fit(X_rf_train, y_rf_train)
    X_todo = df_c.select(NUM_RF).fill_null(0.0).to_numpy()
    hojas = rf.apply(X_todo)
    COLS_HOJA = [f'hoja_arbol_{i}' for i in range(hojas.shape[1])]
    df_c = df_c.with_columns([pl.Series(c, hojas[:, i].astype(str)) for i, c in enumerate(COLS_HOJA)])
    print(f'  {len(COLS_HOJA)} hojas de RF agregadas ({X_rf_train.shape[0]:,} filas de fit).   [{time.time()-t0:.0f}s]')

    # --- poda por shadow features, dos rondas, mayoria de votos ---
    rng = np.random.default_rng(PARAM['semilla'])
    N_SHADOW, N_REP, UMBRAL = PARAM['n_shadow'], PARAM['n_repeticiones_shadow'], PARAM['umbral_mayoria_shadow']
    _sup = df_c.filter(pl.col('target_nivel').is_not_null())
    _periodos_sup = sorted(_sup['periodo'].unique().to_list())
    _val_periodos = set(_periodos_sup[-2:])
    _tr_shadow = _sup.filter(~pl.col('periodo').is_in(_val_periodos))
    _va_shadow = _sup.filter(pl.col('periodo').is_in(_val_periodos))
    _n_tr_shadow = _tr_shadow.height

    def _cols_ruido(n, prefijo):
        return [pl.Series(f'{prefijo}_{i}', rng.normal(size=n) if i % 2 == 0 else rng.uniform(-1, 1, size=n))
               for i in range(N_SHADOW)]

    def _sobreviven_por_shadow(features_candidatas, cats_candidatas, prefijo_shadow):
        conteos = {c: 0 for c in features_candidatas}
        _min_hoja = max(20, _n_tr_shadow // 40)
        for rep in range(N_REP):
            cols_shadow = [f'{prefijo_shadow}{rep}_{i}' for i in range(N_SHADOW)]
            tr = _tr_shadow.with_columns(_cols_ruido(_tr_shadow.height, f'{prefijo_shadow}{rep}'))
            va = _va_shadow.with_columns(_cols_ruido(_va_shadow.height, f'{prefijo_shadow}{rep}'))
            pool = features_candidatas + cols_shadow
            cats_pool = [c for c in cats_candidatas if c in pool]

            def _a_pandas_pool(df_pl):
                out = (df_pl.select(pool + ['target_nivel'])
                            .with_columns([pl.col(c).cast(pl.Utf8).cast(pl.Categorical) for c in cats_pool])
                            .to_pandas())
                for c in cats_pool:
                    out[c] = out[c].astype('category')
                return out

            df_pd_tr, df_pd_va = _a_pandas_pool(tr), _a_pandas_pool(va)
            for c in cats_pool:
                df_pd_va[c] = df_pd_va[c].cat.set_categories(df_pd_tr[c].cat.categories)

            modelo = lgb.LGBMRegressor(objective='regression', metric='mae', verbosity=-1,
                                       n_estimators=100, learning_rate=0.05, num_leaves=15,
                                       min_child_samples=_min_hoja, reg_alpha=1.0, reg_lambda=1.0,
                                       seed=PARAM['semilla'] + rep, n_jobs=-1)
            modelo.fit(df_pd_tr[pool], df_pd_tr['target_nivel'], categorical_feature=cats_pool,
                      eval_set=[(df_pd_va[pool], df_pd_va['target_nivel'])],
                      callbacks=[lgb.early_stopping(20, verbose=False)])
            imp = pd.Series(modelo.booster_.feature_importance(importance_type='gain'), index=pool)
            piso = float(imp[cols_shadow].max())
            for c in features_candidatas:
                if imp.get(c, 0.0) > piso:
                    conteos[c] += 1
        return [c for c in features_candidatas if conteos[c] / N_REP > UMBRAL], conteos

    FEATURES_SOBREVIVEN_BASE, _ = _sobreviven_por_shadow(FEATURES_BASE, CAT_BASE, '_shadowA')
    FEATURES_SOBREVIVEN_HOJA, _ = _sobreviven_por_shadow(COLS_HOJA, COLS_HOJA, '_shadowB')
    FEATURES_FINALES = FEATURES_SOBREVIVEN_BASE + FEATURES_SOBREVIVEN_HOJA
    if not FEATURES_FINALES:
        raise RuntimeError('Ninguna feature sobrevivio la poda por shadow.')
    print(f'  poda: {len(FEATURES_FINALES)} sobreviven de {len(FEATURES_BASE) + len(COLS_HOJA)} candidatas.   [{time.time()-t0:.0f}s]')

    COLS_SIEMPRE = [c for c in (COLS_ID + COLS_TARGET + COLS_META) if c in df_c.columns]
    df_final = df_c.select(COLS_SIEMPRE + FEATURES_FINALES)
    df_train_final = df_final.filter(~pl.col('_es_infer')).drop('_es_infer')
    df_infer_final = df_final.filter(pl.col('_es_infer')).drop('_es_infer')

    guardar_parquet_atomico(df_train_final, PARAM['path_fe_bis'])
    guardar_parquet_atomico(df_infer_final, PARAM['path_fe_bis_infer'])

print(f'[FE avanzado] train {df_train_final.shape}   inferencia {df_infer_final.shape}   [{time.time()-t0:.0f}s]')


## 5) Etapa 3 — Optuna (identico a `03_Optuna.ipynb`)


In [ ]:
t0 = time.time()

df_pd_all = df_train_final.to_pandas()
TIPO_TARGET = PARAM['tipo_target']
TARGET_COL = 'target_delta' if TIPO_TARGET == 'delta' else 'target_nivel'

COLS_EXCLUIR_BASE = ['agrupa_id', 'product_id', 'customer_id', 'periodo', 'tn', 'tn_t2',
                     'target_nivel', 'target_delta', 'modo_agrupacion', 'solo_predecir', 'tipo_target']
FEATURES = [c for c in df_train_final.columns if c not in COLS_EXCLUIR_BASE and c not in PARAM['features_excluir']]
CAT_FEATURES = [c for c in PARAM['cols_categoricas'] if c in FEATURES] + \
              [c for c in FEATURES if c.startswith('hoja_arbol_')]
for c in CAT_FEATURES:
    df_pd_all[c] = df_pd_all[c].astype('category')

print(f'[Optuna] FEATURES: {len(FEATURES)}   CAT_FEATURES: {len(CAT_FEATURES)}')


def calcular_metrica(y_real, y_pred, metrica='wape'):
    y_real = np.array(y_real, dtype=np.float64)
    y_pred = np.maximum(np.array(y_pred, dtype=np.float64), 0.0)
    if metrica == 'wape':
        den = y_real.sum()
        return np.nan if den == 0 else np.abs(y_real - y_pred).sum() / den
    elif metrica == 'mae':
        return np.abs(y_real - y_pred).mean()
    raise ValueError(f'Metrica desconocida: {metrica}')


def sumar_meses(periodo, n):
    anio, mes = divmod(periodo, 100)
    total = (anio * 12 + (mes - 1)) + n
    return (total // 12) * 100 + (total % 12) + 1


def get_splits(periodos, esquema, k=3, mes_obj=2):
    if esquema == 'ultimo_periodo':
        return [(periodos[-2], periodos[-1])]
    elif esquema == 'walk_forward_k':
        return [(periodos[-(i + 1)], periodos[-i]) for i in range(k, 0, -1)]
    elif esquema == 'febreros':
        pset = set(periodos)
        splits = [(p, p) for p in periodos
                 if sumar_meses(p, PARAM['horizonte']) % 100 == mes_obj and sumar_meses(p, PARAM['horizonte']) in pset]
        if not splits:
            print('aviso: No se pudo armar validacion por febreros; usando ultimo_periodo')
            return [(periodos[-2], periodos[-1])]
        return splits
    raise ValueError(f'Esquema desconocido: {esquema}')


periodos_ordenados = sorted(df_pd_all['periodo'].unique().tolist())
splits = get_splits(periodos_ordenados, PARAM['esquema_val'], PARAM['walk_forward_k'], PARAM.get('mes_objetivo', 2))
print(f'[Optuna] splits ({PARAM["esquema_val"]}): {len(splits)}')

if Path(PARAM['path_hiper']).exists() and 'optuna' not in PARAM['forzar']:
    print(f"[Optuna] cache encontrada -> se saltea: {PARAM['path_hiper']}")
    with open(PARAM['path_hiper']) as f:
        cfg_hiper = json.load(f)
else:
    print('[Optuna] corriendo busqueda...')

    def calcular_pesos(periodos_serie, decay):
        if decay is None:
            return None
        periodos = sorted(periodos_serie.unique())
        idx = {p: i for i, p in enumerate(periodos)}
        n = len(periodos)
        return periodos_serie.map(lambda p: decay ** (n - 1 - idx[p])).values

    def espacio_hiper(trial):
        base = {'objective': PARAM['objective_lgbm'], 'metric': 'mae', 'verbosity': -1,
               'boosting_type': 'gbdt', 'seed': PARAM['semilla'], 'subsample_freq': 1}
        if PARAM['regularizacion'] == 'fuerte':
            base.update({
                'num_leaves': trial.suggest_int('num_leaves', 8, 64),
                'max_depth': trial.suggest_int('max_depth', 3, 7),
                'learning_rate': trial.suggest_float('learning_rate', 5e-3, 0.1, log=True),
                'n_estimators': trial.suggest_int('n_estimators', 100, 800),
                'min_child_samples': trial.suggest_int('min_child_samples', 30, 200),
                'subsample': trial.suggest_float('subsample', 0.5, 0.9),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.9),
                'reg_alpha': trial.suggest_float('reg_alpha', 0.1, 20.0, log=True),
                'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 20.0, log=True),
            })
        else:
            base.update({
                'num_leaves': trial.suggest_int('num_leaves', 20, 300),
                'max_depth': trial.suggest_int('max_depth', 3, 12),
                'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
                'n_estimators': trial.suggest_int('n_estimators', 100, 2000),
                'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
                'subsample': trial.suggest_float('subsample', 0.5, 1.0),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
                'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
                'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
            })
        if PARAM['objective_lgbm'] == 'tweedie' and PARAM.get('tweedie_optimizar', False):
            base['tweedie_variance_power'] = trial.suggest_float('tweedie_variance_power', 1.1, 1.9)
        return base

    def objective(trial):
        params = espacio_hiper(trial)
        errores = []
        for corte, val_p in splits:
            df_tr = df_pd_all[df_pd_all['periodo'] < val_p].copy()
            df_vl = df_pd_all[df_pd_all['periodo'] == val_p].copy()
            if len(df_vl) == 0:
                continue
            if PARAM['sampling_frac'] is not None:
                df_tr = df_tr.sample(frac=PARAM['sampling_frac'], random_state=PARAM['semilla'])
            X_tr, y_tr = df_tr[FEATURES], df_tr[TARGET_COL].values
            X_vl, y_vl = df_vl[FEATURES], df_vl[TARGET_COL].values
            w_tr = calcular_pesos(df_tr['periodo'], PARAM['decay_recencia'])
            modelo = lgb.LGBMRegressor(**params)
            modelo.fit(X_tr, y_tr, sample_weight=w_tr, eval_set=[(X_vl, y_vl)],
                      categorical_feature=CAT_FEATURES,
                      callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
            pred = modelo.predict(X_vl)
            if TIPO_TARGET == 'delta':
                tn_actual = df_vl['tn'].values
                pred_nivel, real_nivel = tn_actual + pred, tn_actual + y_vl
            else:
                pred_nivel, real_nivel = pred, y_vl
            errores.append(calcular_metrica(real_nivel, pred_nivel, PARAM['metrica']))
        return float(np.mean(errores))

    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=PARAM['semilla']),
                                study_name=PARAM['study_name'], storage=PARAM['path_storage'], load_if_exists=True)
    print(f'  trials previos: {len(study.trials)}   corriendo {PARAM["n_trials"]} nuevos')
    with tqdm(total=PARAM['n_trials'], desc='Optuna') as pbar:
        def callback(study, trial):
            pbar.update(1)
            pbar.set_postfix({'mejor': f'{study.best_value:.4f}'})
        study.optimize(objective, n_trials=PARAM['n_trials'], callbacks=[callback])
    print(f'  mejor {PARAM["metrica"]} (nivel): {study.best_value:.4f}')

    cfg_hiper = {
        'experimento': PARAM['experimento'], 'modo_agrupacion': PARAM['modo_agrupacion'],
        'metrica': PARAM['metrica'], 'mejor_valor': study.best_value, 'n_trials_total': len(study.trials),
        'esquema_val': PARAM['esquema_val'], 'tipo_target': TIPO_TARGET,
        'objective_lgbm': PARAM['objective_lgbm'], 'regularizacion': PARAM['regularizacion'],
        'decay_recencia': PARAM['decay_recencia'], 'features': FEATURES, 'cat_features': CAT_FEATURES,
        'hiperparametros': study.best_params,
    }
    with open(PARAM['path_hiper'], 'w') as f:
        json.dump(cfg_hiper, f, indent=2)

    db_local = PARAM['path_storage'].replace('sqlite:///', '')
    db_bucket = str(DIR_OUT / f"z303_optuna_{MODO}_{PARAM['experimento']}.db")
    shutil.copy(db_local, db_bucket)

print(f"[Optuna] {cfg_hiper['metrica']} = {cfg_hiper['mejor_valor']:.4f}   [{time.time()-t0:.0f}s]")


## 6) Etapa 4 — Entrenamiento final + submit (identico a `04_Entrenamiento_final.ipynb`)


In [ ]:
t0 = time.time()
if Path(PARAM['path_submit']).exists() and 'final' not in PARAM['forzar']:
    print(f"[final] cache encontrada -> se saltea: {PARAM['path_submit']}")
    tb_submit = pl.read_csv(PARAM['path_submit'])
else:
    print('[final] entrenando modelo final...')
    FEATURES_F = cfg_hiper['features']
    CAT_FEATURES_F = cfg_hiper.get('cat_features', [])
    TIPO_TARGET_F = cfg_hiper.get('tipo_target', 'nivel')
    TARGET_COL_F = 'target_delta' if TIPO_TARGET_F == 'delta' else 'target_nivel'
    hiper = cfg_hiper['hiperparametros']

    tb_pred = pl.read_csv(DIR_RAW / 'product_id_apredecir201912.txt', separator='\t',
                          schema_overrides={'product_id': pl.Int32})

    df_train_pd = df_train_final.to_pandas()
    df_infer_pd = df_infer_final.to_pandas()
    if PARAM['sampling_frac'] is not None:
        df_train_pd = df_train_pd.sample(frac=PARAM['sampling_frac'], random_state=PARAM['semilla'])
    faltantes = [f for f in FEATURES_F if f not in df_train_pd.columns]
    if faltantes:
        FEATURES_F = [f for f in FEATURES_F if f in df_train_pd.columns]
    for c in CAT_FEATURES_F:
        if c in df_train_pd.columns:
            df_train_pd[c] = df_train_pd[c].astype('category')
            df_infer_pd[c] = df_infer_pd[c].astype('category')

    X_train, y_train = df_train_pd[FEATURES_F], df_train_pd[TARGET_COL_F].values
    X_infer = df_infer_pd[FEATURES_F]

    OBJECTIVE_LGBM = cfg_hiper.get('objective_lgbm', 'regression')

    def calcular_pesos_final(periodos_serie, decay):
        if decay is None:
            return None
        periodos = sorted(periodos_serie.unique())
        idx = {p: i for i, p in enumerate(periodos)}
        n = len(periodos)
        return periodos_serie.map(lambda p: decay ** (n - 1 - idx[p])).values

    w_train = calcular_pesos_final(df_train_pd['periodo'], PARAM['decay_recencia'])
    modelos = []
    for s in PARAM['semillas_ensemble']:
        params_lgbm = {'objective': OBJECTIVE_LGBM, 'metric': 'mae', 'verbosity': -1,
                       'boosting_type': 'gbdt', 'seed': s, **hiper}
        m = lgb.LGBMRegressor(**params_lgbm)
        m.fit(X_train, y_train, sample_weight=w_train, categorical_feature=CAT_FEATURES_F)
        modelos.append(m)
    print(f'  ensemble de {len(modelos)} modelo(s) entrenado.   [{time.time()-t0:.0f}s]')

    preds = np.column_stack([m.predict(X_infer) for m in modelos])
    y_pred = preds.mean(axis=1)
    if TIPO_TARGET_F == 'delta':
        y_pred = df_infer_pd['tn'].values + y_pred
    if PARAM['desescalar'] and 'media_rolling' in df_infer_pd.columns:
        media = df_infer_pd['media_rolling'].values
        y_pred = y_pred * np.where(media > 0, media, 1.0)
    y_pred = np.maximum(y_pred, PARAM['clip_min'])

    df_pred = pl.DataFrame({
        'product_id': df_infer_pd['product_id'].values.astype('int32'),
        'periodo': df_infer_pd['periodo'].values.astype('int32'),
        'tn': y_pred.astype('float64'),
    })
    ultimo_p = int(df_infer_pd['periodo'].max())
    df_pred_p = df_pred.filter(pl.col('periodo') == ultimo_p)
    if MODO == 'cliente_producto':
        df_pred_final = df_pred_p.group_by('product_id').agg(pl.col('tn').sum())
    else:
        df_pred_final = df_pred_p.drop('periodo')

    tb_base = tb_pred.with_columns(pl.lit(0.0).alias('tn'))
    tb_submit = (tb_base.join(df_pred_final, on='product_id', how='left', suffix='_pred')
                       .with_columns(pl.coalesce(['tn_pred', 'tn']).alias('tn'))
                       .select(['product_id', 'tn']).sort('product_id'))
    tb_submit.write_csv(PARAM['path_submit'])
    print(f'  submit: {tb_submit.height} productos.   Guardado: {PARAM["path_submit"]}')

print(f'[final] listo.   [{time.time()-t0:.0f}s]')
print(tb_submit.describe())


## 7) Submit a Kaggle (opcional)


In [ ]:
if not PARAM['submit']:
    print("PARAM['submit'] = False -> no se sube. El CSV ya esta generado.")
else:
    mensaje = f"pipe_unico {PARAM['experimento']} | {cfg_hiper['metrica']}={cfg_hiper['mejor_valor']:.4f}"
    res = subprocess.run(
        ['kaggle', 'competitions', 'submit', '-c', PARAM['kaggle_competition'],
         '-f', PARAM['path_submit'], '-m', mensaje],
        capture_output=True, text=True
    )
    print('stdout:', res.stdout)
    print('stderr:', res.stderr)
    print('returncode:', res.returncode)


## 8) Resumen


In [ ]:
print(f"experimento         : {PARAM['experimento']}")
print(f"modo_agrupacion      : {PARAM['modo_agrupacion']}")
print(f"usar_fe_avanzado     : {PARAM['usar_fe_avanzado']}")
print(f"tipo_target          : {PARAM['tipo_target']}")
print(f"objective_lgbm       : {PARAM['objective_lgbm']}")
print(f"regularizacion       : {PARAM['regularizacion']}")
print(f"esquema_val          : {PARAM['esquema_val']}")
print(f"{cfg_hiper['metrica']} (val, interno) : {cfg_hiper['mejor_valor']:.4f}")
print(f"submit               : {PARAM['path_submit']}")
